In [3]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import * 

In [5]:
spark = (
    SparkSession.builder
    .appName("Gold Layer")
    .getOrCreate()
)
spark 

In [7]:
silver_df = spark.read.parquet("../Data/02_Silver/weather_cleaned") 

In [8]:
silver_df.show(5,truncate=False) 

+-------------------+-----------+--------+------------+------------+-------------------------+
|weather_time       |temperature|humidity|weather_date|weather_hour|ingestion_date           |
+-------------------+-----------+--------+------------+------------+-------------------------+
|2026-06-09 00:00:00|30.7       |66      |2026-06-09  |0           |2026-06-16 12:02:07.12629|
|2026-06-09 01:00:00|30.6       |65      |2026-06-09  |1           |2026-06-16 12:02:07.12629|
|2026-06-09 02:00:00|31.3       |63      |2026-06-09  |2           |2026-06-16 12:02:07.12629|
|2026-06-09 03:00:00|32.4       |58      |2026-06-09  |3           |2026-06-16 12:02:07.12629|
|2026-06-09 04:00:00|33.9       |52      |2026-06-09  |4           |2026-06-16 12:02:07.12629|
+-------------------+-----------+--------+------------+------------+-------------------------+
only showing top 5 rows


### Weather Summary ### 

In [18]:
daily_summary = (
    silver_df
        .groupBy("weather_date")
        .agg(
            avg("temperature").alias("avg_temperature"),
            max("temperature").alias("max_temperature"),
            min("temperature").alias("min_temperature"),
            avg("humidity").alias("avg_humidity"),
        )
)

In [12]:
daily_summary.show(5, truncate=False) 

+------------+------------------+---------------+---------------+------------------+
|weather_date|avg_temperature   |max_temperature|min_temperature|avg_humidity      |
+------------+------------------+---------------+---------------+------------------+
|2026-06-20  |33.94166666666666 |40.3           |29.1           |50.333333333333336|
|2026-06-09  |34.916666666666664|41.3           |30.6           |50.916666666666664|
|2026-06-18  |33.92916666666667 |39.1           |29.7           |51.958333333333336|
|2026-06-21  |33.416666666666664|39.4           |29.6           |54.375            |
|2026-06-10  |34.762499999999996|41.5           |30.4           |52.541666666666664|
+------------+------------------+---------------+---------------+------------------+
only showing top 5 rows


### Hourly Temperature Trends ###

In [13]:
hourly_trend = (
    silver_df
    .groupBy("weather_hour")
    .agg(
        avg("temperature").alias("avg_temperature")
    )
    .orderBy("weather_hour")
)

In [16]:
hourly_trend.show(24) 

+------------+------------------+
|weather_hour|   avg_temperature|
+------------+------------------+
|           0|29.914285714285715|
|           1|29.907142857142855|
|           2|30.607142857142854|
|           3| 31.70714285714286|
|           4| 33.16428571428571|
|           5| 34.65714285714286|
|           6| 36.05714285714286|
|           7| 37.23571428571428|
|           8| 38.36428571428571|
|           9| 39.24285714285714|
|          10| 39.85000000000001|
|          11|40.128571428571426|
|          12| 39.29285714285715|
|          13|              37.5|
|          14|35.535714285714285|
|          15| 33.99285714285715|
|          16| 32.97142857142857|
|          17|              32.4|
|          18| 32.06428571428571|
|          19|31.685714285714283|
|          20|31.378571428571433|
|          21|30.935714285714287|
|          22|30.478571428571428|
|          23|30.071428571428577|
+------------+------------------+



### Extreme Weather Report ###

In [19]:
extreme_weather = (
    daily_summary
    .orderBy(
        daily_summary.max_temperature.desc()
    )
)

In [20]:
extreme_weather.show() 

+------------+------------------+---------------+---------------+------------------+
|weather_date|   avg_temperature|max_temperature|min_temperature|      avg_humidity|
+------------+------------------+---------------+---------------+------------------+
|  2026-06-12|           34.3375|           42.1|           29.8|55.666666666666664|
|  2026-06-11|           34.4875|           41.6|           30.2|55.958333333333336|
|  2026-06-10|34.762499999999996|           41.5|           30.4|52.541666666666664|
|  2026-06-09|34.916666666666664|           41.3|           30.6|50.916666666666664|
|  2026-06-13|34.650000000000006|           40.4|           29.7|             49.75|
|  2026-06-20| 33.94166666666666|           40.3|           29.1|50.333333333333336|
|  2026-06-16| 33.86666666666667|           39.9|           29.5|52.291666666666664|
|  2026-06-14| 34.01666666666666|           39.7|           29.5|49.916666666666664|
|  2026-06-15|34.112500000000004|           39.6|           29.8|

### Save the Data In GOLD ### 

In [25]:
daily_summary.write.mode("overwrite").parquet(
    "../Data/03_Gold/daily_weather_summary"
)

hourly_trend.write.mode("overwrite").parquet(
    "../Data/03_Gold/hourly_temperature_trend"
)

extreme_weather.write.mode("overwrite").parquet(
    "../Data/03_Gold/extreme_weather_report"
)